In [ ]:
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.mixture import GaussianMixture

from utils import (
    border_mask, mahalanobis, threshold_value, close, largest_component_filled,
    mask_to_rect, segment, metrics, prefilter_image
)

BEST_PARAMS = {
    "prefilter": "none",
    "color_space": "HSV",
    "channels": [1, 2],
    "threshold": "otsu",
    "close_frac": 0.012306
}

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
FIG_DIR = Path("results/segmentation")
FIG_DIR.mkdir(parents=True, exist_ok=True)
METHODS = ["otsu", "triangle", "gmm", "wallthr"]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

def rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def show(ax, image, title=None, **kwargs):
    ax.imshow(image, **kwargs)
    ax.set_axis_off()
    if title:
        ax.set_title(title)

def gauss(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))

def black_background(mask):
    return np.where(mask.astype(bool), 255, 0).astype(np.uint8)


In [ ]:
# Pipeline parameters: edit these values to choose the image and segmentation settings.
DATA_DIR = Path().resolve().parent / "data" / "qsd2_w2"
IMG_NAME = "00021.jpg"

P = dict(BEST_PARAMS)
# Optional overrides, for example:
# P.update({"prefilter": "median", "color_space": "HSV", "threshold": "gmm"})
# Supported keys and values follow BEST_PARAMS in utils.py.

img_bgr = cv2.imread(str(DATA_DIR / IMG_NAME))
if img_bgr is None:
    raise FileNotFoundError(f"Image not found: {DATA_DIR / IMG_NAME}")
gt_img = cv2.imread(str((DATA_DIR / IMG_NAME).with_suffix(".png")), cv2.IMREAD_GRAYSCALE)
gt = None if gt_img is None else (gt_img > 127).astype(np.uint8)
h, w = img_bgr.shape[:2]

filtered = {name: __import__("utils").prefilter_image(img_bgr, name) for name in ["none", "median"]}
img_f = filtered[P["prefilter"]]
border = border_mask(h, w)
hsv = cv2.cvtColor(img_f, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(img_f, cv2.COLOR_BGR2LAB)
color = {"HSV": hsv, "LAB": lab}[P["color_space"]]
CH = P["channels"]
dist = mahalanobis(color, border, CH)
T = {method: threshold_value(dist, border, method) for method in METHODS}
RAW = {method: (dist > T[method]).astype(np.uint8) for method in METHODS}
print(f"Image: {IMG_NAME} | prefilter: {P['prefilter']} | color space: {P['color_space']} | threshold: {P['threshold']}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show(axes[0], rgb(img_bgr), "Original")
show(axes[1], black_background(gt) if gt is not None else np.zeros((h, w), np.uint8), "Ground truth", cmap="gray", vmin=0, vmax=255)
save(fig, "01_original_ground_truth")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show(axes[0], rgb(img_bgr), "Original")
show(axes[1], rgb(img_f), "Median blur" if P["prefilter"] == "median" else "Filtered")
save(fig, "02_prefilter")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
border_view = rgb(img_bgr).copy()
border_view[border.astype(bool)] = (255, 255, 255)
show(axes[0], border_view, "Border samples")
show(axes[1], black_background(border), "Border mask", cmap="gray", vmin=0, vmax=255)
save(fig, "03_border")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))

for row, (space, data, names) in enumerate([
    ("HSV", hsv, ["H", "S", "V"]),
    ("Lab", lab, ["L*", "a*", "b*"]),
]):
    for col, name in enumerate(names):
        channel = data[..., col]

        vmax = 180 if space == "HSV" and name == "H" else 255

        axes[row, col].imshow(
            channel,
            cmap="gray",
            vmin=0,
            vmax=vmax,
            interpolation="nearest"
        )
        axes[row, col].set_title(name)
        axes[row, col].axis("off")

plt.tight_layout()
save(fig, "04_hsv_lab")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))

for ax, (i, name) in zip(axes, enumerate(["H", "S", "V"])):
    vmax = 180 if name == "H" else 255
    ax.imshow(hsv[..., i], cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
save(fig, "04_hsv")
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax0 = fig.add_subplot(1, 3, 1)
show(ax0, rgb(img_bgr), "Original")
ax1 = fig.add_subplot(1, 3, 2)
im = ax1.imshow(dist, cmap="magma", vmin=0, vmax=np.percentile(dist, 99))
ax1.set_title("Mahalanobis 2D")
ax1.set_axis_off()
ax2 = fig.add_subplot(1, 3, 3, projection="3d")
step = max(1, int(max(h, w) / 100))
yy, xx = np.mgrid[0:h:step, 0:w:step]
dd = dist[::step, ::step]
ax2.plot_surface(xx, yy, dd, cmap="magma", linewidth=0, antialiased=False)
ax2.set_title("Mahalanobis 3D")
ax2.set_xticks([]); ax2.set_yticks([]); ax2.set_zticks([])
save(fig, "05_mahalanobis")
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter1d

GMM_SMOOTH_SIGMA = 4
GMM_SAMPLE_SIZE = 64

hist_max = max(float(np.percentile(dist, 99.7)), max(T.values()), 1.0)
hist, edges = np.histogram(dist.ravel(), bins=180, range=(0, hist_max), density=True)
centers = (edges[:-1] + edges[1:]) / 2
sample = cv2.resize(dist, (GMM_SAMPLE_SIZE, GMM_SAMPLE_SIZE), interpolation=cv2.INTER_CUBIC).reshape(-1, 1).astype(np.float64)
gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=0).fit(sample)
order = np.argsort(gmm.means_.ravel())
g_means = gmm.means_.ravel()[order]
g_sds = np.sqrt(gmm.covariances_.reshape(-1)[order])
g_weights = gmm.weights_[order]
xs = np.linspace(0, hist_max, 700)

# Minimum of the smoothed histogram between the two component means.
s_hist, s_edges = np.histogram(dist.ravel(), bins=256, range=(0, hist_max), density=True)
s_hist = gaussian_filter1d(s_hist.astype(np.float64), sigma=GMM_SMOOTH_SIGMA)
s_centers = (s_edges[:-1] + s_edges[1:]) / 2
between = (s_centers > g_means[0]) & (s_centers < g_means[1])
if not between.any():   # the two gaussians are too close
    T["gmm"] = float((g_means[0] + g_means[1]) / 2)
else:
    T["gmm"] = float(s_centers[between][np.argmin(s_hist[between])])
RAW["gmm"] = (dist > T["gmm"]).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, method_pair in enumerate([["otsu", "triangle"], ["gmm", "wallthr"]]):
    for method in method_pair:
        col = (0 if method == method_pair[0] else 2)
        ax = axes[row, col]
        ax.plot(centers, hist, color="0.35", lw=1.2)
        t = T[method]
        ax.axvline(t, color="black", ls="--", lw=1.5)
        if method == "gmm":
            for i in range(2):
                ax.plot(xs, g_weights[i] * gauss(xs, g_means[i], g_sds[i]), lw=1.8, zorder=3)
            ax.plot(s_centers, s_hist, color="red", lw=1.5, ls="--")   # filtered histogram
            ax.axvline(T["gmm"], color="black", ls="--")
        elif method == "wallthr":
            wall_values = dist[border.astype(bool)]
            ax.axvline(wall_values.mean(), color="tab:blue", ls=":")
            ax.axvline(wall_values.mean() + wall_values.std(), color="tab:orange", ls=":")
        ax.set_title(method.title())
        ax.set_xlim(0, hist_max)
        ax.set_yticks([])
        ax.set_xlabel("Distance")

        mask_ax = axes[row, col + 1]
        show(mask_ax, RAW[method] * 255, None, cmap="gray", vmin=0, vmax=255)
save(fig, "06_thresholds")
plt.show()

In [ ]:
t = T[P["threshold"]]
m_threshold = (dist > t).astype(np.uint8)
m_morph = close(m_threshold, P["close_frac"])
m_component = largest_component_filled(m_morph)
m_rect = mask_to_rect(m_component)
ys, xs = np.where(m_rect)
x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()

fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
show(axes[0], m_threshold * 255, "Threshold", cmap="gray", vmin=0, vmax=255)
show(axes[1], m_morph * 255, "Morphology", cmap="gray", vmin=0, vmax=255)
show(axes[2], m_component * 255, "Component", cmap="gray", vmin=0, vmax=255)
show(axes[3], rgb(img_bgr), "Rectangle")
axes[3].add_patch(Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0 + 1, y1 - y0 + 1,
                            facecolor=(1, 1, 1, 1)))
save(fig, "07_postprocessing")
plt.show()

In [ ]:
from matplotlib.patches import Patch
from utils import PROJ_FRAC

H, W = m_component.shape
col_prof, row_prof = m_component.mean(axis=0), m_component.mean(axis=1)
col_thr, row_thr = PROJ_FRAC * col_prof.max(), PROJ_FRAC * row_prof.max()
ys, xs = np.where(m_rect)
x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()

kept = (m_component == 1) & (m_rect == 1)
discarded = (m_component == 1) & (m_rect == 0)
GREEN, RED = (0.12, 0.75, 0.2), (0.9, 0.12, 0.12)

overlay = rgb(img_bgr).astype(float) / 255
overlay[kept] = 0.5 * overlay[kept] + 0.5 * np.array(GREEN)
overlay[discarded] = 0.5 * overlay[discarded] + 0.5 * np.array(RED)

fig = plt.figure(figsize=(6, 6 * H / W))
gs = fig.add_gridspec(2, 2, width_ratios=[6, 1], height_ratios=[1, 6], hspace=0.03, wspace=0.03)
ax_top, ax_img, ax_right = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

# Image
ax_img.imshow(overlay, aspect="auto")
ax_img.add_patch(Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0 + 1, y1 - y0 + 1,
                           fill=False, ec="white", lw=1, ls=(0, (3, 2))))
ax_img.set_axis_off()

# Column projection
x = np.arange(W)
ax_top.fill_between(x, col_prof, where=col_prof > col_thr, color=GREEN, lw=0)
ax_top.fill_between(x, col_prof, where=col_prof <= col_thr, color=RED, lw=0)
ax_top.axhline(col_thr, color="gray", ls=":", lw=1)
ax_top.set_xlim(0, W); ax_top.set_ylim(0, 1.05)
ax_top.set_axis_off()

# Row projection
y = np.arange(H)
ax_right.fill_betweenx(y, row_prof, where=row_prof > row_thr, color=GREEN, lw=0)
ax_right.fill_betweenx(y, row_prof, where=row_prof <= row_thr, color=RED, lw=0)
ax_right.axvline(row_thr, color="gray", ls=":", lw=1)
ax_right.set_ylim(H, 0); ax_right.set_xlim(0, 1.05)
ax_right.set_axis_off()

fig.legend(handles=[Patch(color=GREEN, label="Kept"), Patch(color=RED, label="Discarded")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.45, -0.04))

save(fig, "08_rectangle_fit")
plt.show()

In [ ]:
COLOR_SPACES = {"HSV": cv2.COLOR_BGR2HSV, "LAB": cv2.COLOR_BGR2LAB}
CHANNEL_NAMES = {"HSV": ["H", "S", "V"], "LAB": ["L*", "a*", "b*"]}

color = cv2.cvtColor(prefilter_image(img_bgr, P["prefilter"]), COLOR_SPACES[P["color_space"]])
names = CHANNEL_NAMES[P["color_space"]]
c_left, c_right = P["channels"]

w = color.shape[1]
half = w // 2
split = np.concatenate([color[:, :half, c_left], color[:, half:, c_right]], axis=1)

t = T[P["threshold"]]
m_threshold = (dist > t).astype(np.uint8)
m_morph = close(m_threshold, P["close_frac"])
m_component = largest_component_filled(m_morph)
m_rect = mask_to_rect(m_component)
ys, xs = np.where(m_rect)
x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()

fig, axes = plt.subplots(1, 7, figsize=(21, 3.5))
show(axes[0], rgb(img_bgr), "Original")
show(axes[1], split, f"{names[c_left]} | {names[c_right]}", cmap="gray", vmin=0, vmax=255)
axes[1].axvline(half - 0.5, color="white", linewidth=1.5)
show(axes[2], dist, "Distance", cmap="magma", vmin=0, vmax=np.percentile(dist, 99))
show(axes[3], m_threshold * 255, "Threshold", cmap="gray", vmin=0, vmax=255)
show(axes[4], m_morph * 255, "Morphology", cmap="gray", vmin=0, vmax=255)
show(axes[5], m_component * 255, "Component", cmap="gray", vmin=0, vmax=255)
show(axes[6], rgb(img_bgr), "Rectangle")
axes[6].add_patch(Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0 + 1, y1 - y0 + 1,
                            facecolor=(1, 1, 1, 1)))
save(fig, "07_postprocessing")
plt.show()